## AN OVERVIEW ON CMI DATASET

### Load dataset

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../datasets/CMI/train.csv") 
compactDF = df.describe(include = 'all')
compactDF

### Filter all non essential columns and preprocess labels

In [ ]:
df          = df.drop(['row_id', 'sequence_counter', 'subject'], axis = 1)
compactDF   = compactDF.drop(['row_id', 'sequence_counter', 'subject'], axis = 1)

targetDF    = df[df['sequence_type'] == "Target"]
nonTargetDF = df[df['sequence_type'] == "Non-Target"]
binaryClass = ["Non-Target", "Target"]
macroClass  = targetDF['gesture'].unique().tolist() + ['Non-Target']

nonTargetDF['gesture'] = 'Non-Target'
df = pd.concat([targetDF, nonTargetDF])
del targetDF, nonTargetDF
print(f"Binary Class Name: {binaryClass}")
print(f"Macro Class Name: {macroClass}")

/tmp/ipykernel_56851/2177466249.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nonTargetDF['gesture'] = 'Non-Target'


Binary Class Name: ['Non-Target', 'Target']
Macro Class Name: ['Cheek - pinch skin', 'Forehead - pull hairline', 'Neck - scratch', 'Neck - pinch skin', 'Eyelash - pull hair', 'Eyebrow - pull hair', 'Forehead - scratch', 'Above ear - pull hair', 'Non-Target']


In [ ]:
categoricalColumns = []
continuousColumns  = []
thermaCofColumns   = []
for columnName in compactDF.columns:
    if pd.isna(compactDF[columnName]['mean']):
        categoricalColumns += [columnName]
    else:
        continuousColumns  += [columnName]
    if columnName.startswith("thm_") or columnName.startswith("tof_"):
        thermaCofColumns   += [columnName]
        
print(f"Categorical Columns - Count: {len(categoricalColumns)}. Name: {categoricalColumns}")
print(f"Continuous Columns - Count: {len(continuousColumns)}. Name: {continuousColumns}.")

Categorical Columns - Count: 6. Name: ['sequence_type', 'sequence_id', 'orientation', 'behavior', 'phase', 'gesture']
Continuous Columns - Count: 332. Name: ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8', 'tof_1_v9', 'tof_1_v10', 'tof_1_v11', 'tof_1_v12', 'tof_1_v13', 'tof_1_v14', 'tof_1_v15', 'tof_1_v16', 'tof_1_v17', 'tof_1_v18', 'tof_1_v19', 'tof_1_v20', 'tof_1_v21', 'tof_1_v22', 'tof_1_v23', 'tof_1_v24', 'tof_1_v25', 'tof_1_v26', 'tof_1_v27', 'tof_1_v28', 'tof_1_v29', 'tof_1_v30', 'tof_1_v31', 'tof_1_v32', 'tof_1_v33', 'tof_1_v34', 'tof_1_v35', 'tof_1_v36', 'tof_1_v37', 'tof_1_v38', 'tof_1_v39', 'tof_1_v40', 'tof_1_v41', 'tof_1_v42', 'tof_1_v43', 'tof_1_v44', 'tof_1_v45', 'tof_1_v46', 'tof_1_v47', 'tof_1_v48', 'tof_1_v49', 'tof_1_v50', 'tof_1_v51', 'tof_1_v52', 'tof_1_v53', 'tof_1_v54', 'tof_1_v55', 'tof_1_v56', 'to

### Preprocessing sequence

In [ ]:
uniqueSubj = np.unique(df['sequence_id'])
sequenceDF  = []; sequenceTarget = []
IMUsequence = []; IMUtarget      = []
total = 0
for idx, (subj, subdf) in enumerate(df.groupby("sequence_id")):
    total        += len(subdf) 
    if subdf[thermaCofColumns].isna().any(axis = None):
        IMUsequence += [subdf.drop("sequence_type", axis = 1)[continuousColumns].values]
        IMUtarget   += [[subdf['sequence_type'].iloc[0], subdf['gesture'].iloc[0]]]
    else:
        sequenceDF     += [subdf.drop("sequence_type", axis = 1)[continuousColumns].values]
        sequenceTarget += [[subdf['sequence_type'].iloc[0], subdf['gesture'].iloc[0]]]

print(f"#Unique Subject: {len(uniqueSubj)}")
print(f"Average seq length: {total / (idx + 1)}")
print(f"#IMU only sequence: {len(IMUsequence)}")
del df

#Unique Subject: 8151
Average seq length: 70.53674395779659
#IMU only sequence: 508


## Training with a simple LSTM or CNN

In [ ]:

from sklearn.model_selection import train_test_split


trainData, _, trainTarget, _ = train_test_split(sequenceDF, sequenceTarget, random_state = 42, stratify = sequenceTarget, shuffle = True, test_size = 0.6)
del sequenceDF, sequenceTarget
trainData, valData, trainTarget, valTarget = train_test_split(trainData, trainTarget, random_state = 42, stratify = trainTarget, shuffle = True, test_size = 0.1)


In [ ]:
import os, sys
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, root)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import BinaryF1Score, F1Score

from sklearn.preprocessing import StandardScaler, LabelEncoder
from utils.helper import EarlyStopping
from tqdm.auto import tqdm

class SimpleLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        num_layers: int,
        num_classes: list[int],
        bidirectional: bool = False,
        dropout: float = 0.0
    ):
        super(SimpleLSTMClassifier, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
    
        num_directions = 2 if self.bidirectional else 1
        self.init_h = nn.Parameter(torch.zeros(num_directions * self.num_layers, 1, hidden_size))
        self.init_c = nn.Parameter(torch.zeros(num_directions * self.num_layers, 1, hidden_size))
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.multi_head = nn.ModuleList([
            nn.Sequential(
                nn.LazyLinear(hidden_size // 2),
                nn.LeakyReLU(inplace = True),
                nn.LazyLinear(head)
            ) for head in num_classes
        ])
        
        self.dropout = nn.Dropout(p=dropout)
        
    def forward(self, x, lengths):
        batch_size = x.size(0)

        h0 = self.init_h.repeat(1, batch_size, 1)
        c0 = self.init_c.repeat(1, batch_size, 1)
        
        packed        = pack_padded_sequence(x, lengths, batch_first = True, enforce_sorted = False)
        out, (hn, cn) = self.lstm(packed, (h0, c0))
        out, _        = pad_packed_sequence(out, batch_first = True)
        
        last_output = out[:, -1, :]
        
        out_drop = self.dropout(last_output)
        
        return tuple(head(out_drop) for head in self.multi_head)

class DataWaiter(Dataset):
    def __init__(self, features, target):
        super(DataWaiter, self).__init__()
        
        self.features = features
        self.target   = target
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return torch.tensor(self.features[index]), self.target[index]
    
    @staticmethod
    def collate_fn(batch):
        sequences, labels = zip(*batch)
        lengths = torch.tensor([len(sequence) for sequence in sequences], dtype = torch.long)
        
        paddedSeq = pad_sequence(sequences, batch_first = True).to(torch.float)
        labels    = torch.stack([torch.tensor(label, dtype = torch.long) for label in labels])
        return paddedSeq, lengths, labels
    


class ScaleSequential:
    def __init__(self):
        self.scaler = StandardScaler()
        
    def transform(self, data):
        return [self.scaler.transform(seq) for seq in data]
    
    def fit(self, data):
        tempData = np.r_[*data]
        self.scaler.fit(tempData)
        
    def fit_transform(self, data):
        self.fit(data)
        return self.transform(data)
    
device = torch.device('cuda')
        

### Scaling data

In [ ]:
trainTarget = np.array(trainTarget)
valTarget   = np.array(valTarget)

leBin   = LabelEncoder(); leBin.classes_   = np.array(binaryClass)
leMacro = LabelEncoder(); leMacro.classes_ = np.array(macroClass)
trainTarget[:, 0] = leBin.fit_transform(trainTarget[:, 0])
valTarget[:, 0]   = leBin.transform(valTarget[:, 0])
trainTarget[:, 1] = leMacro.fit_transform(trainTarget[:, 1])
valTarget[:, 1]   = leMacro.transform(valTarget[:, 1])

trainTarget = trainTarget.astype(int)
valTarget   = valTarget.astype(int)

            
scaler    = ScaleSequential()
trainData = scaler.fit_transform(trainData)
valData   = scaler.transform(valData)

### Loading into dataloader

In [ ]:
trainDS = DataWaiter(trainData, trainTarget)
trainLoader = DataLoader(trainDS, batch_size = 64, shuffle = True, num_workers = 12, persistent_workers = True, pin_memory = True, collate_fn = trainDS.collate_fn)

paddedSeq, lengths, _ = next(iter(trainLoader))
paddedSeq.shape # B, SeqLength, F

# -1 on macroc class since "No-target" class is already in binary
model = SimpleLSTMClassifier(input_size = 332, hidden_size = 512, num_layers = 10, num_classes = [1, len(macroClass) - 1], bidirectional = True).to(device)
model(paddedSeq.to(device), lengths)

writeRoot = f"../runs/CMI/{model}"

In [ ]:
epochs = 50; initLR = 2e-4; finalLR = 1e-6; l1 = 1e-4; l2 = 1e-4
lambdaCont = 5.0; lambdaCat = 1.0

binaryCriterion = nn.BCEWithLogitsLoss()
macroCriterion  = nn.CrossEntropyLoss(label_smoothing = 0.1)
optimizer       = optim.AdamW(model.parameters(), lr = initLR)
scheduler       = optim.lr_scheduler.CosineAnnealingLR(optimizer = optimizer, T_max = epochs, eta_min = finalLR)
earlyStop       = EarlyStopping(patience = 20, path = f"{writeRoot}/pretrainedBest.pt", verbose = True)
binaryF1        = F1Score('binary', average = 'none').to(device)
macroF1         = F1Score('multiclass', num_classes = len(macroClass), average = 'macro').to(device)

In [ ]:

pbar = tqdm(range(epochs), desc="Training Epochs", position = 0)
for epoch in pbar:
    model.train()
    
    trainBar = tqdm(trainLoader, desc = "Train", position = 1, leave = False)
    trainMetrics = {"Loss": 0, "Supervised": 0, "Accuracy": 0, "Correct": 0, "Samples": 0, "AUC": 0}
    for sequence, lengths, target in trainBar:
        
        optimizer.zero_grad()
        sequence  = sequence.to(device, non_blocking = True)
        target    = target.to(device, non_blocking = True)
        batchSize = sequence.shape[0]

        binLogits, macroLogits = model(sequence, lengths)
        binaryLoss = binaryCriterion(binLogits, target[:, 0].unsqueeze(1).float())
        print(True)
        binClass   = torch.sigmoid(binLogits); print(binClass)
        macroLoss  = macroCriterion(macroLogits, target[:, 1])


        weightParams = [p for n, p in model.named_parameters()
                        if p.requires_grad and "weight" in n]
        l1Norm = sum(p.abs().sum() for p in weightParams)
        l2Norm = sum(p.pow(2.0).sum() for p in weightParams)
        

        loss = binaryLoss \
            + macroLoss \
            + l1 * l1Norm \
            + l2 * l2Norm


    #     trainMetrics["Samples"]    += batchSize
    #     trainMetrics["Loss"]       += loss.item()
    #     trainMetrics["Supervised"] += supervisedLoss.item()
    #     trainMetrics["Correct"]    += correct

    #     trainBar.set_postfix({
    #         "Loss": f"{trainMetrics['Loss']/(trainBar.n+1):.3f}",
    #         "Supervised": f"{trainMetrics['Supervised']/(trainBar.n+1):.3f}",
    #         "Correct": f"{trainMetrics['Correct']}/{trainMetrics['Samples']}",
    #         "Accuracy": f"{100 * trainMetrics['Correct'] / trainMetrics['Samples']:.2f}%",
    #     })
        
        loss.backward()
        optimizer.step()
        
        
    # trainMetrics["Supervised"] /= len(trainLoader)
    # trainMetrics["Loss"]       /= len(trainLoader)
    # trainMetrics["Accuracy"]   = 100 * trainMetrics["Correct"] / trainMetrics["Samples"]
    # trainMetrics["AUC"]        = auc.compute().item()
    # auc.reset()
    

    # with torch.no_grad():
    #     valBar = tqdm(valLoader, desc = "Val", position = 2, leave = False)
    #     valMetrics = {"Supervised": 0, "Accuracy": 0, "Samples": 0, "Correct": 0, "AUC": 0}
    #     for xCont, xCat, target in valBar:
    #         xCont     = xCont.to(device, non_blocking = True)
    #         xCat      = xCat.to(device, non_blocking = True)
    #         target    = target.to(device, non_blocking = True)
    #         batchSize = xCont.shape[0]


    #         logits         = model(xCont, xCat)
    #         correct        = (logits.argmax(dim=1) == target).sum().item()
    #         supervisedLoss = classCriterion(logits, target)
    #         auc.update(torch.softmax(logits, dim=1), target)


    #         valMetrics["Samples"]    += batchSize
    #         valMetrics["Supervised"] += supervisedLoss.item()
    #         valMetrics["Correct"]    += correct

    #         valBar.set_postfix({
    #             "Supervised": f"{valMetrics['Supervised'] / (valBar.n+1):.3f}",
    #             "Accuracy": f"{100 * valMetrics['Correct'] / valMetrics['Samples']:.2f}%",
    #             "Correct": f"{valMetrics['Correct']}/{valMetrics['Samples']}",
    #         })

    #     valMetrics['Supervised'] /= len(valLoader)
    #     valMetrics['Accuracy']   = 100 * valMetrics['Correct'] / valMetrics['Samples']
    #     valMetrics['AUC']        = auc.compute().item()
    #     auc.reset()

    # scheduler.step()
    # currentLr = optimizer.param_groups[0]['lr']
    
    # used     = torch.cuda.memory_allocated()  / 2**20
    # reserved = torch.cuda.memory_reserved()   / 2**20

    # tqdm.write(
    #     f"Epoch {epoch+1}/{epochs} — "
    #     f"Sup Train: {trainMetrics['Supervised']:.4f}, "
    #     f"Acc Train: {trainMetrics['Accuracy']:.2f}%, "
    #     f"AUC Train: {trainMetrics['AUC']:.4f}, "
    #     f"Sup Val: {valMetrics['Supervised']:.4f}, "
    #     f"Acc Val: {valMetrics['Accuracy']:.2f}%, "
    #     f"AUC Val: {valMetrics['AUC']:.4f}, "
    #     f"No update: {earlyStop.counter}/{earlyStop.patience}"
    # )

    # writer.add_scalar("Loss/Supervised Train",  trainMetrics["Supervised"], epoch+1)
    # writer.add_scalar("Loss/Supervised Val",    valMetrics["Supervised"],   epoch+1)
    # writer.add_scalar("Metrics/Train/Accuracy", trainMetrics["Accuracy"],   epoch+1)
    # writer.add_scalar("Metrics/Train/AUC",      trainMetrics["AUC"],        epoch+1)
    # writer.add_scalar("Metrics/Val/Accuracy",   valMetrics["Accuracy"],     epoch+1)
    # writer.add_scalar("Metrics/Val/AUC",        valMetrics["AUC"],          epoch+1)
    # writer.add_scalar("Misc/LearningRate",      currentLr,                  epoch+1)
    # writer.add_scalar("Misc/Memory/Allocated",  used,                       epoch+1)
    # writer.add_scalar("Misc/Memory/Reserved",   reserved,                   epoch+1)
    # writer.flush()

    # earlyStop(valMetrics['Supervised'], model)
    # if earlyStop.early_stop:
    #     tqdm.write("Early stopping triggered.")
    #     break


Training Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

Train:   0%|          | 0/43 [00:00<?, ?it/s]

### Discovered somethings
- This is a tabular data with time series embedded
- A floating point mean on sequence_counter suggest that the sequence between any subject is not equal => Using LSTM or GRU as an extractor
- Goal is to differentiate BFRB (8 types) from normal actions (10 types)
### Noting for model
- The model must not consist of 20 model ensembles
- Must generalizes well as the metrics are prone to shakeup
### Notes from competition

- The dataset appears huge with a 1.2 GB size but we have approx. 5100 sequences of data here in the training set. This translates to an effective data size of 5100 rows, making this a small data challenge
- Past CMI competitions, especially the PIU challenge was known for its shakeups. Single models and very simple approaches prevailed while large and complex ensembles failed - this could be a directive for this competition too
- The metric is prone to shakeup as a few well classified points will alter the score considerably while a few poor classifications could reduce the score considerably. Be careful to design simple models that generalize well rather than a lot of complex models with questionable generalization potential
- This is a code competition with an API - beginners transitioning with the playground competitions may find the API a bit different. Please follow the standard submission kernel as a guideline and edit this for starters (https://www.kaggle.com/code/sohier/cmi-2025-demo-submission)
- Remember to test the prediction values on a dummy train set before submitting, especially if you are new to this type of competitions - erroneous submissions are counted in the daily quota, so be careful and submit correctly
- Start off with simple single models and make way for FE - this is crucial here. We don't need a baroque ensemble here- simple ML models seem to be sufficient if done well. Usually, leaders and toppers focus on FE in such competitions while the rest focus on model training - I am sure one's direction is clear here.
- Finally, keep track of your work with any experiment tracking tool. It could be a simple excel sheet, a tool like W&B, Neptune.ai, MLOps tools like Kedro / MLFlow, etc. Experiment tracking is a good way to orient yourself with regard to ideas that worked and did not work and revisit your work later in the competition. Usually these artefacts are useful while teaming up as well as one could articulate the work done with aplomb.
- Save the fitted models and OOF predictions as you move along and keep them handy for later use - this trick will surely help you at the end of the competition when you may want to blend and fuse submissions and models for just an extra bit of alpha
